# 红黑树

{{ video_embed | replace("%%VID%%", "uOKHuYloCsI")}}

正如我们已经看到的，哈希表是实现映射 ADT 的一种高效数据结构。
它们提供摊销的、预期的常数时间性能&mdash;这是一个微妙的保证，
因为必须加上“摊销”和“预期”这两个限定词。哈希表还需要可变性来实现。
作为函数式程序员，我们更愿意尽可能避免可变性。

那么，让我们研究如何实现函数式映射。最好的数据结构之一是*红黑树*：
它是一种平衡二叉搜索树，提供最坏情况下的对数时间性能。
一方面，它的性能比哈希表稍差（对数时间而非常数时间）；
另一方面，我们不必再用“摊销”和“预期”这样的词限定性能。
对数时间即使面对很大的工作负载也仍然非常高效。而且，我们可以避免可变性！

## 二叉搜索树

{{ video_embed | replace("%%VID%%", "Xx9V5vrkjJA")}}

**二叉搜索树** (BST) 是具有以下内容的二叉树
表示不变式：

> 对于任何节点 *n*，*n* 左子树中的每个节点的值都小于
> *n* 的值，并且 *n* 右子树中的每个节点的值都大于
> 大于 *n* 的值。

我们称之为*BST 不变量*。

下面是在 BST 上实现几个操作的代码：

In [ ]:
type 'a tree = Node of 'a * 'a tree * 'a tree | Leaf

(** [mem x t] is [true] iff [x] is a member of [t]. *)
let rec mem x = function
  | Leaf -> false
  | Node (y, l, r) ->
    if x < y then mem x l
    else if x > y then mem x r
    else true

(** [insert x t] is [t] with [x] inserted as a member. *)
let rec insert x = function
  | Leaf -> Node (x, Leaf, Leaf)
  | Node (y, l, r) as t ->
    if x < y then Node (y, insert x l, r)
    else if x > y then Node (y, l, insert x r)
    else t

这些操作的运行时间是多少？由于 `insert` 只是一个 `mem`
通过额外的恒定时间节点创建，我们专注于 `mem` 操作。

{{ video_embed | replace("%%VID%%", "pFUJgaYUH6o")}}

`mem`的运行时间为$O(h)$，其中$h$是树的高度，
因为每个递归调用都会在树中下降一层。什么是
最坏情况下树的高度？它发生在一个由 $n$ 节点组成的树上
长分支&mdash;想象将数字1,2,3,4,5,6,7按顺序添加到
树。所以 `mem` 最坏情况下的运行时间仍然是 $O(n)$，其中 $n$ 是
树中的节点数。

对于允许快速查找的树来说，什么是好的形状？
一棵“完美二叉树”，其中每个叶子都位于底层，对于其中的节点数量而言，其高度可能最小。
考虑一下这棵完美的树：
```ocaml
Node (2, Node (1, Leaf, Leaf), Node (3, Leaf, Leaf))
```
我们可以这样画它，其中 `.` 表示 `Leaf`，我们描述树的每一层的高度：
```
                     Height
        2              2
       / \
     1     3           1
    / \   / \
   .   . .   .         0
```

完美二叉树中的节点数 $n$ 与树的高度 $h$ 呈指数关系。
（请注意，这里的“节点”具体指的是 `Node` 而不是 `Leaf`。）
特别是 $n = 2^h - 1$ （这是我们可以通过归纳法证明的事实）。
因此$h = \log_2(n+1)$，即$O(\log n)$。
因此，完美二叉搜索树上的查找操作将以对数时间运行。

{{ video_embed | replace("%%VID%%", "BGkipOJdH3U")}}

当然，并不是每棵树都是完美的。
但如果我们能够使树中所有路径的长度接近对数，那么我们仍然可以获得良好的性能。
*平衡*树数据结构的作用是：当添加或删除元素时，树的形状可能会发生变化，以确保路径长度不会增长太长。
平衡二叉搜索树数据结构的一些示例（按照对路径长度的约束从最强到最弱的顺序排列）包括：

- 2-3 棵树（Hopcroft，1970）：所有路径都具有相同的长度。
- AVL 树（Adelson-Velsky 和 Landis，1962）：最短和最长路径的长度最多相差 1。
- 红黑树（Guibas 和 Sedgewick，1978）：最短和最长路径的长度最多相差两倍。

这些数据结构中的每一个都确保 $O(\log n)$ 运行时间。
接下来我们将研究红黑树，它有一个基于代数数据类型和模式匹配的特别好的实现。

## 红黑树

{{ video_embed | replace("%%VID%%", "JzhG0jDxGqg")}}

红黑树是比较简单的平衡二叉树数据结构。
我们将树的每个节点着色为*红色*或*黑色*。

In [ ]:
type color = Red | Black
type 'a rbtree = Leaf | Node of color * 'a * 'a rbtree * 'a rbtree

请注意，当我们在本次讨论中写“节点”时，我们指的是 `Node` 构造函数； **a `Leaf` 不是节点。**
我们要求非空树的根节点被涂成黑色。

除了二叉搜索树表示不变量之外，红黑树还必须满足这些不变量：

1. **局部不变式：** 任何路径上都不存在两个相邻的红色节点。

2. **全局不变式：** 每个“完整路径”（从根到叶子的路径）都具有相同数量的黑色节点。这个数字称为树的“黑高度”。

如果一棵树满足这两个条件，那么该树的每个子树也必须满足这两个条件。
如果子树违反了任一条件，则整棵树也会违反。

### 平衡

红黑树不变量确保树保持平衡。
平衡不一定是完美的——一些完整路径可能比其他完整路径更长——但正如以下定理所述，不平衡不会太严重。

**定理** 红黑树中最长全路径的长度至多是最短全路径长度的两倍。

*证明。*
根据全局不变量，最长和最短的完整路径必须具有相同数量的黑色节点 $b$，即树的黑色高度。
（如果最长完整路径或最短完整路径存在联系，则没关系。考虑这些联系中的任何路径。）
如果最短的节点没有红色节点，则其长度仅为 $b$。
根据局部不变量，通过添加 $b$ 红色节点，最长的节点至多可以使长度加倍。
例如，假设最短完整路径是 B-B-B-B-Leaf，其中“B”表示黑色节点。
其长度为四。
考虑到树的黑色高度，最长的完整路径是多少？
这将是一条在每个黑色节点之后添加一个红色节点的路径，即 B-R-B-R-B-R-B-R-Leaf，其中“R”表示红色节点。
它的长度是八，是最短长度的两倍。 &#x25A1;

### 成员查询

{{ video_embed | replace("%%VID%%", "gDTCRj2-bCU")}}

我们检查红黑树中的成员资格的方式与二叉搜索树完全相同：

In [ ]:
let rec mem x = function
  | Leaf -> false
  | Node (_, y, l, r) ->
    if x < y then mem x l
    else if x > y then mem x r
    else true

`mem` 的效率是多少？
在最坏的情况下，它会递归到树中最低的叶子，并且找不到元素 `x` ——也就是说，运行时间由树的高度决定。
红黑树的高度是节点数量的对数，如下定理成立。

**定理。** 具有 $n$ 节点的红黑树的高度 $h$ 最多为 $O(\log n)$。

*证明。*
令 $b$ 为树的黑色高度，即每条完整路径上黑色节点的数量。
我们将使用 $b$ 在以下两个引理中建立 $h$ 和 $n$ 之间的关系。

引理 1：$h \leq 2b$。 
考虑长度为 $h$ 的节点的任何完整路径。
路径上必须有 $b$ 黑色节点。
如果存在一些红色节点，那么正如我们在上面关于平衡的证明中所论证的那样，最多可以有 $b$ 个。
因此，该路径的长度最多为 $2b$。
因此，$h \leq 2b$。

引理 2：$n \geq 2^b - 1$。
考虑树中的黑色节点。
由于每个完整路径必须具有 $b$ 黑色节点，因此我们可以从这些节点中形成高度为 $b$ 的完美树。
（从根开始，它是黑色的。保留它。递归到每个子节点。如果子节点是黑色的，则保留它。如果它是红色的并且没有子节点，则将其丢弃。如果它是红色的并且有子节点，则这些子节点必须是黑色的；任意选择一个来替换红色节点并丢弃另一个黑色子节点。）
如上所述，完美的树至少有 $2^b - 1$ 个节点。
因此，$n \geq 2^b - 1$。

最后：我们可以将引理2重写如下：

$$
& n \geq 2^b - 1 \\
& \equiv 2^b \leq n + 1 \\
& \equiv b \leq \log_2(n + 1) \\
& \equiv 2b \leq 2 \log_2(n + 1)
$$

然后利用引理 1，我们可以将该不等式左侧的 $2b$ 的下界放宽到 $h$，从而获得 $h \leq 2 \log_2(n + 1)$。
因此 $h$ 至多是 $O(\log n)$。 &#x25A1;

### 插入：冈崎算法

更有趣的是 `insert` 操作。与
标准二叉树，我们通过替换搜索找到的叶子来添加一个节点
程序。但是我们可以给这个节点上什么颜色呢？

- 将其涂成黑色可能会增加该路径的黑色高度，从而违反了
全局不变式。

- 将其着色为红色可能会使其与另一个红色节点相邻，从而违反了
局部不变量。

所以总的来说，这两种选择都不安全。 Chris Okasaki（*纯函数数据
Structures*，1999）给出了一个优雅的算法，通过选择解决问题
违反局部不变量，然后走上树来修复违规。
这是它的工作原理。

{{ video_embed | replace("%%VID%%", "dCBAhbIEoYM")}}

我们总是将新节点涂成红色以确保全局不变式
保存下来。如果新节点的父节点已经是黑色，则局部不变式
没有被侵犯。在这种情况下，我们就完成了插入：有
没有违规，也不需要修复这棵树。一个常见的案例在
这种情况发生在新节点的父节点是树的根时，即
已经被染成黑色了。

但如果新节点的父节点是红色的，则违反了局部不变式。
在这种情况下，新节点的父节点不能是树的根（黑色），
因此新节点有一个祖父母。爷爷奶奶一定是黑人
因为局部不变量在我们插入新节点之前保持不变。现在我们有
恢复局部不变量需要做的工作。

{{ video_embed | replace("%%VID%%", "igUOhpGICgA")}}

下图显示了可能出现的四种可能的违规行为。其中，
`a`-`d` 可能是空子树，`x`-`z` 是存储在节点处的值。
节点颜色用 `R` 和 `B` 指示。我们已经标记了较低的
两个带有方括号的违规红色节点。当我们开始修复这棵树时
标记的节点将是我们刚刚插入的新节点。因此它会有
没有孩子 &mdash; 例如，在情况 1 中， `a` 和 `b` 将是叶子。 （稍后
不过，当我们走上树继续修复时，我们可能会遇到
标记节点具有非空子树的情况。）

```text
           1             2             3             4

           Bz            Bz            Bx            Bx
          / \           / \           / \           / \
         Ry  d         Rx  d         a   Rz        a   Ry
        /  \          / \               /  \          /  \
     [Rx]  c         a  [Ry]          [Ry]  d        b   [Rz]
     /  \               /  \          / \                /  \
    a    b             b    c        b   c              c    d
```

请注意，在每棵树中，我们都仔细标记了值和节点
这样 BST 不变量可确保以下排序：

```text
all nodes in a
 <
  x
   <
    all nodes in b
     <
      y
       <
        all nodes in c
         <
          z
           <
            all nodes in d
```

因此，我们可以通过替换来变换树以恢复局部不变量
上述四种情况中的任何一种：

```text
         Ry
        /  \
      Bx    Bz
     / \   / \
    a   b c   d
```

{{ video_embed | replace("%%VID%%", "D4FJMJUIMSw")}}

这种转换被一些作者称为“旋转”。将 `y` 视为
是树的一种轴或中心。所有其他节点和子树
作为旋转的一部分围绕它移动。冈崎称这种转变为
*平衡*操作。将其视为改善树的平衡，就像你
可以看到上面最终的树的形状与原来的四棵树的比较
案例。这个平衡函数可以使用模式简单简洁地编写
匹配，其中四个输入情况中的每一个都映射到相同的输出情况。
另外，还存在树在本地保持不变的情况。

In [ ]:
let balance = function
  | Black, z, Node (Red, y, Node (Red, x, a, b), c), d
  | Black, z, Node (Red, x, a, Node (Red, y, b, c)), d
  | Black, x, a, Node (Red, z, Node (Red, y, b, c), d)
  | Black, x, a, Node (Red, y, b, Node (Red, z, c, d)) ->
    Node (Red, y, Node (Black, x, a, b), Node (Black, z, c, d))
  | a, b, c, d -> Node (a, b, c, d)

*为什么旋转（即平衡操作）可以保持 BST 不变式？*
检查上面的数字，让自己相信旋转的树可以确保
所有节点和子树的正确排序。选择哪些标签
放置在第一个图中的位置很聪明，并且保证了
最终树在所有四种情况下都具有相同的标签。

*为什么旋转会保留全局不变性？*在旋转之前，树
满足全局不变量。这意味着子树 `a`-`d` 下面
祖父母都有相同的黑色高度，祖父母加一
高度。在旋转后的树中，子树都处于同一级别，但现在 `x`
并 `z` 在该高度上加一。树的整体黑色高度没有
改变，并且每条路径继续具有相同的黑色高度。

*为什么旋转会建立局部不变量？* 唯一的局部不变量
旋转之前树中涉及标记节点的违规。之后
轮换后，该违规行为已被消除。此外，由于 `x` 和 `z` 是
旋转后颜色为黑色，它们无法创建新的局部不变量
违反子树 `a`-`d` 的根（如果有）。然而，该问题的根源
旋转后的树现在是 `y` 并且颜色为红色。如果该节点有一个父节点&mdash;that
也就是说，如果情况 1-4 中的祖父母不是整个
tree&mdash;那么我们可能刚刚创建了一个新的局部不变违规
`y` 与其父级之间！

为了解决可能出现的新违规问题，我们需要继续沿着树向上走
从 `y` 到根，并修复进一步的局部不变违规。在
最坏的情况下，该过程会一直级联到树的顶部，并且
结果产生两个相邻的红色节点，其中之一刚刚成为根。但如果
发生这种情况时，我们只需将这个新根从红色重新着色为黑色即可。这样就完成了
恢复局部不变量。它还保留了全局不变量，同时
将整棵树的总黑色高度增加一个&mdash;，即
插入时黑色高度增加的唯一方法。 `insert` 代码
使用 `balance` 如下：

In [ ]:
let insert x s =
  let rec ins = function
    | Leaf -> Node (Red, x, Leaf, Leaf)
    | Node (color, y, a, b) as s ->
      if x < y then balance (color, y, ins a, b)
      else if x > y then balance (color, y, a, ins b)
      else s
  in
  match ins s with
  | Node (_, y, a, b) -> Node (Black, y, a, b)
  | Leaf -> (* guaranteed to be non-empty *)
    failwith "RBT insert failed with ins returning leaf"

`insert` 完成的工作量是 $O(\log n)$。它以 `ins` 向下递归
树到叶子，这是插入发生的地方，然后调用 `balance`
备份过程中的每一步。到叶子的路径长度为$O(\log n)$，
因为树已经平衡了。并且，每次调用 `balance` 都是 $O(1)$
工作。

{{ video_embed | replace("%%VID%%", "giSzhfuTMMA")}}

### 删除

从红黑树中删除元素是有效的
类似地。我们从删除 BST 元素开始，然后进行重新平衡。当
内部（非叶）节点被删除，如果它的数量较少，我们只需将其拼接出来
多于两个非叶子节点；如果它有两个非叶孩子，我们找到下一个
树中的值，必须在其右子节点中找到。

但是，在从红黑树中移除期间平衡树需要考虑
更多案例。从树中删除黑色元素会产生以下可能性：
树中的某些路径的黑色节点太少，破坏了全局不变性。

Germane 和 Might 发明了一种优雅的算法来处理这种重新平衡
解决方案是创建“双黑”节点，在确定
黑色高度。欲了解更多信息，请阅读他们的论文：[*删除：红黑的诅咒
Tree* *函数式编程杂志*][gm]，第 24 卷，第 4 期，2014 年 7 月。

[gm]: https://doi.org/10.1017/S0956796814000227

## 来自 BST 的映射和集合

{{ video_embed | replace("%%VID%%", "B_e8Qr4nl4A")}}

使用 BST 来实现映射或集合 ADT 很容易：

- 对于映射，只需在每个节点存储一个绑定即可。节点的排序方式为
键。这些值与顺序无关。

- 对于集合，只需在每个节点存储一个元素即可。节点的排序方式为
元素。

OCaml 标准库对 `Map` 和 `Set` 模块执行此操作。它使用一个
平衡 BST 是 AVL 树的变体。 AVL 树是平衡 BST
路径的高度最多允许变化 1。 OCaml 标准
库修改了它以允许高度最多变化 2。就像红黑一样
树，它们实现了最坏情况的对数性能。

现在我们有了一个函数式映射数据结构，它与我们的映射数据结构相比如何？
命令式版本，哈希表？

- **持久性：** 我们的红黑树是持久性的，但是哈希表是持久性的
短暂的。

- **性能：** 我们得到保证的最坏情况对数性能
红黑树，但采用散列表摊销、预期恒定时间。
  考虑到涉及的所有修饰符，这有点难以比较。这也是一个
  持久数据结构通常必须的普遍现象的示例
  与等效的临时数据结构相比，需要支付额外的对数成本。

- **方便：** 我们必须提供平衡二进制的排序函数
树和哈希表的哈希函数。大多数库都提供了默认值
  为了方便起见，使用哈希函数。但是哈希表的性能确实
  依赖于散列函数真正在桶上随机分配键。如果
  事实并非如此，哈希表性能保证的“预期”部分
  被侵犯。所以便利性是一把双刃剑。

这里没有明显的赢家。由于 OCaml 库同时提供 `Map` 和
`Hashtbl`，你可以选择。